In [7]:
import nibabel as nib
import numpy as np
import random
import matplotlib.pyplot as plt
import os
import pandas as pd

In [8]:
 conductivity_ranges = {
        # Original tissues
        'White-Matter': [0.08, 0.3], #wenger
        'Gray-Matter': [0.15, 0.5], #wenger
        'CSF': [1.64, 1.94], #wenger
        'Bone': [0.003, 0.032], #wenger
        'Scalp': [0.14, 0.45], #wenger 
        'Eye_balls': [0.5, 1.5], # timmons #Opitz
        'Compact_bone': [0.003, 0.01], # M. Akhtari
        'Spongy_bone': [0.016, 0.041], # M. Akhtari
        'Blood': [0.6, 0.7], #gabriel 2009
        'Muscle': [0.22, 0.36], #gabriel 2009
        'Electrode': [1e6, 1e6], 
        'gel': [0.1, 0.3], #wenger #Lang Peritumoral Edema
        
        # Tumor regions
        'Necrosis': [0.5, 1.5], #wenger
        'Edema': [0.4, 1.5], # LangPeritumoral Edema
        'Enhanced_Tumor': [0.15, 0.5], #wenger
        
    }

    # Define tumor label mapping (tumor seg uses 1=Edema, 2=Enhanced, 3=Necrosis)
    #assign new labels starting from 1000 to avoid conflicts
tumor_mapping = {
    1: 1001,  # Necrosis
    2: 1002,  # Edema
    3: 1003   #Enhanced Tumor
}

 # Define all label mppings (original + tumor)
all_labels = {
    # Original labels (from your first lookup table)
    1: conductivity_ranges['White-Matter'],
    2: conductivity_ranges['Gray-Matter'],
    3: conductivity_ranges['CSF'],
    4: conductivity_ranges['Bone'],
    5: conductivity_ranges['Scalp'],
    6: conductivity_ranges['Eye_balls'],
    7: conductivity_ranges['Compact_bone'],
    8: conductivity_ranges['Spongy_bone'],
    9: conductivity_ranges['Blood'],
    10: conductivity_ranges['Muscle'],
    2000: conductivity_ranges['Electrode'],
    2400: conductivity_ranges['gel'],
    2001: conductivity_ranges['Electrode'],
    2401: conductivity_ranges['gel'],
    
    # Tumor labels
    tumor_mapping[1]: conductivity_ranges['Necrosis'],
    tumor_mapping[2]: conductivity_ranges['Edema'],
    tumor_mapping[3]: conductivity_ranges['Enhanced_Tumor']
}

voltage_mapping = {
    2000: -1,
    2001: 1
}

In [9]:
def merge_segmentations(base_path, tumor_path):
        base = nib.load(base_path)
        tumor = nib.load(tumor_path)
        
        # Verify dimensions match
        #if base.shape != tumor.shape:
            #raise ValueError("Segmentation dimensions don't match!")
        
        base_data = base.get_fdata()
        tumor_data = tumor.get_fdata()
        
        # Create merged segmentation (copy base first)
        merged = base_data.copy()
        
        
        
        # Replace tumor labels in merged data
        for old_label, new_label in tumor_mapping.items():
            merged[tumor_data == old_label] = new_label
        
        
        #merged_img = nib.Nifti1Image(merged, base.affine , base.header)
        #nib.save(merged_img, output_dir)
        return merged

In [10]:
 # Step 3: Create conductivity maps
def create_conductivity_map(seg_data, output_path , base_seg):
        cond_data = np.zeros_like(seg_data)
        
       
        
        # Assign conductivities
        for label, (low, high) in all_labels.items():
            mask = seg_data == label
            cond_data[mask] = random.uniform(low, high)
        
        
        # Save conductivity map
        cond_img = nib.Nifti1Image(cond_data ,nib.load(base_seg).affine, nib.load(base_seg).header)
        nib.save(cond_img, output_path)
        
        return cond_data


In [11]:
def create_vlotage_map(base_seg_path, output_path):
    labeled_image = nib.load(base_seg_path)
    labeled_data = labeled_image.get_fdata()

    voltage_data = np.zeros_like(labeled_data)
    
    for label, voltage in voltage_mapping.items():
        mask = labeled_data == label
        voltage_data[mask] = voltage


    voltage_image = nib.Nifti1Image(voltage_data, labeled_image.affine , labeled_image.header)
    nib.save(voltage_image, output_path)

    return voltage_data

In [12]:
df = pd.read_csv("E:/Datasets/Stage 2/add electrode folder/All Subjects .csv")
subject_id_column = 'Subject ID'
for subject_id in df[subject_id_column]:
    # Define paths (change these to your actual file paths)
    tumor_seg_path = f"E:/Datasets/Re-Creation/Co_registration/{subject_id}/seg/seg_to_mni.nii.gz"
    output_base_dir = "E:/Datasets/Stage 2/Conductivity maps"
    output_dir = os.path.join(output_base_dir, f"{subject_id}")
    
    voltage_output_base_dir = "E:/Datasets/Stage 2/voltage"
    voltage_output_dir = os.path.join(voltage_output_base_dir, f"{subject_id}")
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(voltage_output_dir, exist_ok=True)
    
    for i in range(1, 6):
        base_seg_path = f"E:/Datasets/Stage 2/labeled nifti with electrodes/{subject_id}/labels_{i}.nii.gz"
        # Perform the merge
        merged_data = merge_segmentations(base_seg_path, tumor_seg_path)
        # Generate 5 conductivity maps
        
        cond_map = create_conductivity_map(
            merged_data,
            os.path.join(output_dir, f'conductivity_map_{i}.nii.gz'),
            base_seg_path
        )
    
        voltage_map = create_vlotage_map(base_seg_path, 
            os.path.join(voltage_output_dir, f'voltage_map_{i}.nii.gz')
        )
        
    print(f"Created 5 conductivity maps in {output_dir}")

Created 5 conductivity maps in E:/Datasets/Stage 2/Conductivity maps\TCGA-02-0006
Created 5 conductivity maps in E:/Datasets/Stage 2/Conductivity maps\TCGA-02-0009
Created 5 conductivity maps in E:/Datasets/Stage 2/Conductivity maps\TCGA-02-0011
Created 5 conductivity maps in E:/Datasets/Stage 2/Conductivity maps\TCGA-02-0027
Created 5 conductivity maps in E:/Datasets/Stage 2/Conductivity maps\TCGA-02-0033
Created 5 conductivity maps in E:/Datasets/Stage 2/Conductivity maps\TCGA-02-0034
Created 5 conductivity maps in E:/Datasets/Stage 2/Conductivity maps\TCGA-02-0037
Created 5 conductivity maps in E:/Datasets/Stage 2/Conductivity maps\TCGA-02-0046
Created 5 conductivity maps in E:/Datasets/Stage 2/Conductivity maps\TCGA-02-0047
Created 5 conductivity maps in E:/Datasets/Stage 2/Conductivity maps\TCGA-02-0054
Created 5 conductivity maps in E:/Datasets/Stage 2/Conductivity maps\TCGA-02-0059
Created 5 conductivity maps in E:/Datasets/Stage 2/Conductivity maps\TCGA-02-0064
Created 5 conduc

In [2]:
help(nib.Nifti1Image)

Help on class Nifti1Image in module nibabel.nifti1:

class Nifti1Image(Nifti1Pair, nibabel.filebasedimages.SerializableImage)
 |  Nifti1Image(dataobj, affine, header=None, extra=None, file_map=None)
 |  
 |  Class for single file NIfTI1 format image
 |  
 |  Method resolution order:
 |      Nifti1Image
 |      Nifti1Pair
 |      nibabel.analyze.AnalyzeImage
 |      nibabel.spatialimages.SpatialImage
 |      nibabel.dataobj_images.DataobjImage
 |      nibabel.filebasedimages.SerializableImage
 |      nibabel.filebasedimages.FileBasedImage
 |      builtins.object
 |  
 |  Methods defined here:
 |  
 |  update_header(self)
 |      Harmonize header with image data and affine
 |  
 |  ----------------------------------------------------------------------
 |  Data and other attributes defined here:
 |  
 |  files_types = (('image', '.nii'),)
 |  
 |  header_class = <class 'nibabel.nifti1.Nifti1Header'>
 |      Class for NIfTI1 header
 |      
 |      The NIfTI1 header has many more coded fie